# Сборка датасета под задачу 1

из нарезанного среза собираем один плоский датасет для рекомендательной сети и предсказываем оценку stars по табличным признакам юзера и заведения

основа это таблица reviews, где одна строка это один отзыв, то есть пара юзер-заведение с известной оценкой. к ней слева подклеиваем признаки business по business_id и users по user_id.

текст отзывов и таблицу tips сюда не берём, это для задачи 2.

In [ ]:
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
from _constants import REVIEWS_PARQUET, BUSINESS_PARQUET, USERS_PARQUET, PROCESSED
TOPK = 20
OUT = PROCESSED / "task1_dataset.parquet"

## Загрузка таблиц среза
из reviews берём только нужные колонки, текст и голоса не грузим, про утечки ниже. переименовываем колонки которые иначе столкнутся при join.

In [ ]:
cols = ["review_id", "user_id", "business_id", "stars", "date"]
reviews = pd.read_parquet(REVIEWS_PARQUET, columns=cols)
business = pd.read_parquet(BUSINESS_PARQUET)
business = business.drop(columns=["name", "postal_code", "city", "state"])
business = business.rename(columns={"stars": "biz_avg_stars", "review_count": "biz_review_count"})
users = pd.read_parquet(USERS_PARQUET)
users = users.rename(columns={"review_count": "user_review_count", "useful": "user_useful", "funny": "user_funny", "cool": "user_cool"})
print(reviews.shape)
print(business.shape)
print(users.shape)

reviews самая большая таблица, business и users поменьше. часть колонок переименовали чтобы они не столкнулись при join

## Объединение в один датасет
reviews это левая таблица, гранулярность сохраняется, одна строка на отзыв.

In [ ]:
df = reviews.merge(business, on="business_id", how="left")
df = df.merge(users, on="user_id", how="left")
print(df.shape)
assert len(df) == len(reviews)
before = len(df)
df = df.dropna(subset=["average_stars"]).reset_index(drop=True)
print("удалено строк без профиля юзера", before - len(df))
df.head(3)

после джойнов это плоская таблица, одна строка на отзыв. строки без профиля юзера выкинули, их немного.

## Чистка утечек и инжиниринг признаков

утечки таргета уже исключены тем что мы не грузили text и голоса отзыва, они известны только после написания отзыва. дальше добавляем несколько признаков. account_age_days это стаж юзера на момент отзыва, разница date и yelping_since. price_range пуст примерно у 44% заведений, поэтому делаем флаг price_known и заполняем пропуски медианой. user_avg_minus_biz показывает насколько юзер в среднем строже или добрее среднего по заведению.

biz_avg_stars, biz_review_count и average_stars это агрегаты по всем отзывам, то есть мягкая утечка, поэтому ниже делаем сплит по времени.

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["yelping_since"] = pd.to_datetime(df["yelping_since"])
df["account_age_days"] = (df["date"] - df["yelping_since"]).dt.days.clip(lower=0)
df["price_known"] = df["price_range"].notna().astype(int)
df["price_range"] = df["price_range"].fillna(df["price_range"].median())
df["user_avg_minus_biz"] = df["average_stars"] - df["biz_avg_stars"]
df = df.drop(columns=["yelping_since"])

добавили стаж аккаунта, флаг известности цены и разницу оценок юзера и заведения. колонку yelping_since убрали.

## Кодирование категориальных признаков

categories это мульти-метка, одно заведение попадает в несколько категорий, поэтому кодируем multi-hot по топ-K самых частых категорий. city_state это всего 3 города, его делаем one-hot.

In [ ]:
cat_sets = df["categories"].fillna("").apply(lambda s: {x.strip() for x in s.split(",") if x.strip()})
top_cats = df["categories"].dropna().str.split(", ").explode().value_counts().head(TOPK).index.tolist()
def slug(name):
    return "cat_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
for cat in top_cats:
    df[slug(cat)] = cat_sets.apply(lambda st: int(cat in st))
cat_cols = [slug(c) for c in top_cats]
city_oh = pd.get_dummies(df["city_state"], prefix="city").astype(int)
df = pd.concat([df, city_oh], axis=1)
city_cols = city_oh.columns.tolist()
df = df.drop(columns=["categories", "city_state"])
print("категорий multi-hot", len(cat_cols))
print("городов one-hot", len(city_cols))

получили 20 бинарных колонок категорий и несколько колонок городов, исходные текстовые поля удалили

## Временной train/val/test сплит
делим по date 70/15/15: учимся на прошлом, проверяем на будущем

In [ ]:
df = df.sort_values("date").reset_index(drop=True)
q70, q85 = df["date"].quantile([0.70, 0.85])
df["split"] = np.where(df["date"] <= q70, "train", np.where(df["date"] <= q85, "val", "test"))
print(df["split"].value_counts(normalize=True).round(3).to_dict())
print(q70.date(), q85.date())

сплит по времени дал примерно 70/15/15. train это прошлое, val и test это будущее, так агрегаты не подсматривают.

## Сохранение готового датасета
итог это один parquet: ключи, date, split, таргет stars и табличные признаки.

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT, index=False)
num_cols = ["biz_avg_stars", "biz_review_count", "is_open", "price_range", "price_known",
            "latitude", "longitude", "user_review_count", "average_stars",
            "user_useful", "user_funny", "user_cool", "fans", "n_friends",
            "n_elite_years", "account_age_days", "user_avg_minus_biz"]
feature_cols = num_cols + cat_cols + city_cols
print(df.shape)
print("признаков", len(feature_cols))

сохранили один parquet. таргет это stars, дальше идут числовые признаки, категории multi-hot и города one-hot, плюс служебные ключи и дата.

## Корреляционный анализ
смотрим линейную связь признаков с stars и между собой по Пирсону, только на train. хотим понять какие признаки несут сигнал, а какие дублируют друг друга.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
train = df[df["split"] == "train"]
corr = train[feature_cols + ["stars"]].corr(method="pearson")
target_corr = corr["stars"].drop("stars").sort_values(ascending=False)
target_corr.round(3)

### Корреляция с целевой переменной

In [ ]:
plt.figure(figsize=(7, 9))
plt.barh(target_corr.index[::-1], target_corr.values[::-1])
plt.title("Корреляция признаков с оценкой")
plt.xlabel("Pearson r")
plt.show()

### Матрица корреляций числовых признаков

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(train[num_cols + ["stars"]].corr(), annot=True, fmt=".2f")
plt.title("Матрица корреляций числовых признаков")
plt.show()

In [ ]:
cm = train[feature_cols].corr().abs()
mask = np.triu(np.ones(cm.shape), k=1).astype(bool)
pairs = cm.where(mask).stack().sort_values(ascending=False)
strong = pairs[pairs > 0.7]
print(strong.round(3))

сильно скоррелированных пар почти нет, значит мультиколлинеарность модели не мешает.

## Заведения с несколькими категориями

у Yelp поле categories это список тегов, где смешаны зонтичные теги типа Restaurants или Food и конкретные кухни типа Mexican или Italian. смотрим два вопроса: насколько заведения мультикатегориальны и сколько сочетают несколько разных кухонь сразу.

In [ ]:
from collections import Counter
from itertools import combinations
import pandas as pd
from _constants import BUSINESS_PARQUET
biz_cat = pd.read_parquet(BUSINESS_PARQUET, columns=["business_id", "categories"])
cat_lists = biz_cat["categories"].fillna("").apply(lambda s: [c.strip() for c in s.split(",") if c.strip()])
n_cat = cat_lists.apply(len)
print("всего заведений", len(biz_cat))
print("среднее число категорий", round(n_cat.mean(), 2))
print("с 2 и более категориями", int((n_cat >= 2).sum()), round((n_cat >= 2).mean(), 3))
print(n_cat.value_counts().sort_index().head(8))

у большинства заведений несколько категорий сразу, поле categories действительно мультилейбл.

### Мульти-кухонные заведения

берём только национальные кухни типа Mexican или Italian. зонтичные теги и форматы блюд типа Pizza сюда не входят. считаем заведения у которых таких кухонь две и больше.

In [ ]:
CUISINES = {
    "Mexican", "Italian", "Chinese", "Japanese", "Thai", "Indian", "Vietnamese", "Korean",
    "Mediterranean", "Greek", "French", "Spanish", "Middle Eastern", "American (Traditional)",
    "American (New)", "Latin American", "Caribbean", "Cuban", "Asian Fusion", "Tex-Mex",
    "Cajun/Creole", "Southern", "Filipino", "Hawaiian", "German", "Irish", "British",
    "Lebanese", "Turkish", "Ethiopian", "Peruvian", "Brazilian", "Portuguese", "Pakistani",
    "Persian/Iranian", "Halal", "Soul Food", "Taiwanese", "Mongolian", "Afghan",
}
cuisine_lists = cat_lists.apply(lambda lst: [c for c in lst if c in CUISINES])
n_cuisine = cuisine_lists.apply(len)
is_food = cat_lists.apply(lambda lst: ("Restaurants" in lst) or ("Food" in lst))
n_food = int(is_food.sum())
multi = n_cuisine >= 2
print("заведений общепита", n_food)
print("с 2 и более кухнями", int(multi.sum()), round(multi.sum()/n_food, 3))
print(n_cuisine[n_cuisine >= 1].value_counts().sort_index())

некоторые кухни часто встречаются вместе, сочетания вполне логичные.

### Самые частые пары кухонь

In [ ]:
pair_counter = Counter()
for lst in cuisine_lists:
    for a, b in combinations(sorted(set(lst)), 2):
        pair_counter[(a, b)] += 1
print(pair_counter.most_common(15))

## Подготовка данных для entity embeddings

для эмбеддингов категории кодируем не one-hot а целыми индексами, сеть сама сделает из индекса вектор. словари строим только на train, индекс 0 это pad или unknown. categories это мультилейбл, кодируем списком индексов с паддингом до фиксированной длины. city_state и business_id это одиночные индексы. числовые признаки прогоняем через log1p и StandardScaler с fit на train, все мэппинги и scaler сохраняем в artifacts. latitude и longitude не берём, они дублируют город, а user_id не берём из-за холодного старта, примерно 61% юзеров с одним отзывом.

In [ ]:
import json
from collections import Counter
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from _constants import PROCESSED, BUSINESS_PARQUET, ARTIFACTS
MIN_CAT_FREQ = 20
MAXLEN = 10
cols = ["review_id", "business_id", "split", "stars",
        "biz_avg_stars", "biz_review_count", "is_open", "price_range", "price_known",
        "user_review_count", "average_stars", "user_useful", "user_funny", "user_cool",
        "fans", "n_friends", "n_elite_years", "account_age_days", "user_avg_minus_biz"]
emb = pd.read_parquet(PROCESSED / "task1_dataset.parquet")[cols].copy()
biz = pd.read_parquet(BUSINESS_PARQUET, columns=["business_id", "categories", "city_state"])
biz["cat_list"] = biz["categories"].fillna("").apply(lambda s: [x.strip() for x in s.split(",") if x.strip()])
emb = emb.merge(biz[["business_id", "cat_list", "city_state"]], on="business_id", how="left")
train_mask = emb["split"] == "train"
print(len(emb), int(train_mask.sum()))

### Словари индексов, fit только на train

In [ ]:
train = emb[train_mask]
train_biz_ids = set(train["business_id"])
train_biz = biz[biz["business_id"].isin(train_biz_ids)]
freq = Counter(c for lst in train_biz["cat_list"] for c in lst)
kept = sorted([c for c, v in freq.items() if v >= MIN_CAT_FREQ])
cat_vocab = {c: i + 1 for i, c in enumerate(kept)}
city_vocab = {c: i + 1 for i, c in enumerate(sorted(train["city_state"].dropna().unique()))}
biz_vocab = {b: i + 1 for i, b in enumerate(sorted(train["business_id"].unique()))}
print("словарь категорий", len(cat_vocab))
print("словарь городов", len(city_vocab))
print("словарь заведений", len(biz_vocab))

словарь категорий небольшой, городов всего несколько, а заведений много. под каждый сделаем свой эмбеддинг.

### Кодирование в индексы и паддинг мультилейбла

In [ ]:
def encode_cats(lst):
    ids = [cat_vocab[c] for c in (lst or []) if c in cat_vocab][:MAXLEN]
    return ids + [0] * (MAXLEN - len(ids))
cat_mat = np.array(emb["cat_list"].apply(encode_cats).to_list(), dtype="int64")
emb["city_idx"] = emb["city_state"].map(city_vocab).fillna(0).astype("int64")
emb["biz_idx"] = emb["business_id"].map(biz_vocab).fillna(0).astype("int64")
val_test = emb[emb["split"] != "train"]
print("oov заведений в val/test", round((val_test['biz_idx'] == 0).mean(), 3))
no_cat = (emb['split'] != 'train').to_numpy()
print("строк без известной категории", round((cat_mat[no_cat].sum(axis=1) == 0).mean(), 3))

часть заведений из val/test не встречалась в train, это холодный старт, такие получают индекс 0 и их закрывает эмбеддинг для unknown.

### Числовые признаки: log1p и StandardScaler

In [ ]:
NUM = ["biz_avg_stars", "biz_review_count", "is_open", "price_range", "price_known",
       "user_review_count", "average_stars", "user_useful", "user_funny", "user_cool",
       "fans", "n_friends", "n_elite_years", "account_age_days", "user_avg_minus_biz"]
LOG = ["biz_review_count", "user_review_count", "user_useful", "user_funny",
       "user_cool", "fans", "n_friends"]
Xnum = emb[NUM].copy()
Xnum[LOG] = np.log1p(Xnum[LOG])
scaler = StandardScaler().fit(Xnum[train_mask.to_numpy()])
Xnum_scaled = pd.DataFrame(scaler.transform(Xnum), columns=["num_" + c for c in NUM])
print(round(Xnum_scaled[train_mask.to_numpy()].mean().abs().max(), 4), round(Xnum_scaled[train_mask.to_numpy()].std().mean(), 3))

после log1p и стандартизации среднее около нуля и std около единицы, числовые признаки готовы для сети.

### Сохранение входа и артефактов

In [ ]:
ARTIFACTS.mkdir(parents=True, exist_ok=True)
json.dump(cat_vocab, open(ARTIFACTS / "task1_vocab_categories.json", "w"), ensure_ascii=False)
json.dump(city_vocab, open(ARTIFACTS / "task1_vocab_city.json", "w"), ensure_ascii=False)
json.dump(biz_vocab, open(ARTIFACTS / "task1_vocab_business.json", "w"), ensure_ascii=False)
joblib.dump(scaler, ARTIFACTS / "task1_scaler.joblib")
out = pd.concat([
    emb[["review_id", "split", "stars", "city_idx", "biz_idx"]].reset_index(drop=True),
    pd.DataFrame(cat_mat, columns=[f"catid_{i}" for i in range(MAXLEN)]),
    Xnum_scaled.reset_index(drop=True),
], axis=1)
out.to_parquet(PROCESSED / "task1_embed_inputs.parquet", index=False)
print(out.shape)

## Утилита кодирования ID для инференса

хелпер грузит словари и scaler из artifacts и не зависит от состояния ноутбука. он делает прямой маппинг строки в индекс, незнакомое значение уходит в 0, обратный маппинг из индекса в строку, и функцию encode_example которая готовит вход модели для одного объекта так же как при обучении.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from _constants import ARTIFACTS
def load_vocab(name):
    with open(ARTIFACTS / f"task1_vocab_{name}.json", encoding="utf-8") as f:
        return json.load(f)
CAT_VOCAB = load_vocab("categories")
CITY_VOCAB = load_vocab("city")
BIZ_VOCAB = load_vocab("business")
SCALER = joblib.load(ARTIFACTS / "task1_scaler.joblib")
def invert(vocab):
    return {i: s for s, i in vocab.items()}
ID2CAT, ID2CITY, ID2BIZ = invert(CAT_VOCAB), invert(CITY_VOCAB), invert(BIZ_VOCAB)
MAXLEN = 10
NUM = ["biz_avg_stars", "biz_review_count", "is_open", "price_range", "price_known",
       "user_review_count", "average_stars", "user_useful", "user_funny", "user_cool",
       "fans", "n_friends", "n_elite_years", "account_age_days", "user_avg_minus_biz"]
LOG = ["biz_review_count", "user_review_count", "user_useful", "user_funny",
       "user_cool", "fans", "n_friends"]
print(len(CAT_VOCAB), len(CITY_VOCAB), len(BIZ_VOCAB))

In [ ]:
def to_idx(value, vocab):
    return vocab.get(value, 0)
def from_idx(i, inv_vocab):
    return inv_vocab.get(i, "<unk>")
def encode_categories(cat_list):
    ids = [CAT_VOCAB[c] for c in (cat_list or []) if c in CAT_VOCAB][:MAXLEN]
    return ids + [0] * (MAXLEN - len(ids))
def encode_example(business_id, city_state, cat_list, num_features):
    x = pd.DataFrame([num_features], columns=NUM).astype(float)
    x[LOG] = np.log1p(x[LOG])
    x_scaled = SCALER.transform(x)[0]
    return {
        "biz_idx": to_idx(business_id, BIZ_VOCAB),
        "city_idx": to_idx(city_state, CITY_VOCAB),
        "cat_ids": encode_categories(cat_list),
        "num": x_scaled.astype("float32"),
    }

### Демонстрация: прямой/обратный маппинг и OOV

In [ ]:
sample_id = next(iter(BIZ_VOCAB))
idx = to_idx(sample_id, BIZ_VOCAB)
print("прямой", sample_id, idx)
print("обратный", from_idx(idx, ID2BIZ))
print("неизвестный id", to_idx("НЕСУЩЕСТВУЕТ", BIZ_VOCAB))
demo_cats = ["Restaurants", "Mexican", "Tex-Mex", "НесуществующаяКухня"]
enc = encode_categories(demo_cats)
print("категории", demo_cats)
print("индексы", enc)
print("расшифровка", [from_idx(i, ID2CAT) for i in enc if i != 0])

прямой и обратный маппинг сходятся, неизвестный id уходит в 0. кодирование работает как надо.

In [ ]:
demo_num = {c: 0.0 for c in NUM}
demo_num.update({"biz_avg_stars": 4.5, "biz_review_count": 120, "is_open": 1,
                 "price_range": 2, "price_known": 1, "user_review_count": 30,
                 "average_stars": 3.8, "fans": 5, "n_friends": 40,
                 "n_elite_years": 1, "account_age_days": 1500, "user_avg_minus_biz": -0.7})
ex = encode_example(sample_id, "Tucson, AZ", ["Restaurants", "Mexican"], demo_num)
print("biz_idx", ex["biz_idx"])
print("city_idx", ex["city_idx"], from_idx(ex["city_idx"], ID2CITY))
print("cat_ids", ex["cat_ids"])
print("num", np.round(ex["num"], 2))

для одного объекта получаем индексы и отмасштабированный вектор признаков, ровно как при обучении.

## Финальная обработка признаков: город, время, выбросы

три операции, все параметры фитятся только на train. город это 3 значения, делаем one-hot. время достаём из date и кодируем циклически через sin и cos, чтобы декабрь был рядом с январём, воскресенье с понедельником, а 23 часа с полуночью. тяжёлые счётчики винзоризуем по краям, нижний и верхний процент на train, потом log1p и StandardScaler.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from _constants import PROCESSED, ARTIFACTS
ff = pd.read_parquet(PROCESSED / "task1_dataset.parquet")
ff["date"] = pd.to_datetime(ff["date"])
tr = ff["split"] == "train"
print(len(ff), int(tr.sum()))

### Кодирование города в one-hot

In [ ]:
city_cols = [c for c in ff.columns if c.startswith("city_")]
print(city_cols)
print(ff.loc[tr, city_cols].sum().astype(int).to_string())
print(bool((ff[city_cols].sum(axis=1) == 1).all()))

в каждой строке ровно один город, one-hot корректный.

### Циклическое кодирование времени

In [ ]:
d = ff["date"].dt
ff["year"] = d.year
ff["is_weekend"] = (d.dayofweek >= 5).astype(int)
def cyclic(values, period):
    ang = 2 * np.pi * values / period
    return np.sin(ang), np.cos(ang)
ff["month_sin"], ff["month_cos"] = cyclic(d.month, 12)
ff["dow_sin"], ff["dow_cos"] = cyclic(d.dayofweek, 7)
ff["hour_sin"], ff["hour_cos"] = cyclic(d.hour, 24)
time_cols = ["year", "is_weekend", "month_sin", "month_cos",
             "dow_sin", "dow_cos", "hour_sin", "hour_cos"]
print(time_cols)
print(ff[["date"] + time_cols].head(3).to_string(index=False))

временные признаки закодированы циклически, теперь декабрь и январь близки по sin/cos.

### Нормализация экстремальных значений

In [ ]:
HEAVY = ["biz_review_count", "user_review_count", "user_useful", "user_funny",
         "user_cool", "fans", "n_friends"]
winsor = {}
for c in HEAVY:
    lo, hi = ff.loc[tr, c].quantile([0.01, 0.99])
    winsor[c] = {"lo": float(lo), "hi": float(hi)}
    ff[c] = ff[c].clip(lo, hi)
LOG = HEAVY
NUM_SCALE = ["biz_avg_stars", "biz_review_count", "price_range", "user_review_count",
             "average_stars", "user_useful", "user_funny", "user_cool", "fans",
             "n_friends", "n_elite_years", "account_age_days", "user_avg_minus_biz", "year"]
ff[LOG] = np.log1p(ff[LOG])
scaler = StandardScaler().fit(ff.loc[tr, NUM_SCALE])
ff[NUM_SCALE] = scaler.transform(ff[NUM_SCALE])
print(round(ff.loc[tr, NUM_SCALE].mean().abs().max(), 4), round(ff.loc[tr, NUM_SCALE].std().mean(), 3))

после винзоризации и log1p тяжёлые хвосты ужались, среднее около нуля и std около единицы.

### Сохранение финальной таблицы и артефактов

In [ ]:
final = ff.drop(columns=["date"])
final.to_parquet(PROCESSED / "task1_features.parquet", index=False)
json.dump(winsor, open(ARTIFACTS / "task1_winsor_bounds.json", "w"), indent=0)
joblib.dump(scaler, ARTIFACTS / "task1_scaler_final.joblib")
cat_cols = [c for c in final.columns if c.startswith("cat_")]
print(final.shape)

финальная таблица сохранена: город, время, числовые признаки и категории. это вход для модели.